# Colab Training & Evaluation Setup
Run this notebook on Google Colab to prepare the repository, install dependencies, and execute object detection training and evaluation.


In [1]:
# Install required Python packages
!pip install -q torch torchvision torchaudio torchmetrics pycocotools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.7 MB/s eta 0:00:0000:01


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!ls "/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO"

annotations  test2017  train2017  val2017


In [3]:
%cd /content

!rm -rf vehicle-damage-triage
!git clone https://github.com/AymanLakhnati/AI-vehicule-damage-triage vehicle-damage-triage

%cd /content/vehicle-damage-triage

/content
Cloning into 'vehicle-damage-triage'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 207 (delta 15), reused 205 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 37.02 MiB | 35.73 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/vehicle-damage-triage


In [5]:
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "per-class-eval-final",
            "metadata": {
                "language": "python"
            },
            "outputs": [],
            "source": [
                "from pathlib import Path\n",
                "\n",
                "checkpoint = Path('/content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth')\n",
                "dataset_root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')\n",
                "test_annotations = dataset_root / 'annotations' / 'instances_test2017.json'\n",
                "test_images = dataset_root / 'test2017'\n",
                "\n",
                "required = {\n",
                "    'Epoch 3 checkpoint': checkpoint,\n",
                "    'test annotations': test_annotations,\n",
                "    'test image directory': test_images,\n",
                "}\n",
                "missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]\n",
                "if missing:\n",
                "    raise FileNotFoundError('Required evaluation files are missing:\\n' + '\\n'.join(missing))\n",
                "\n",
                "print(f'Checkpoint: {checkpoint}')\n",
                "print(f'Test annotations: {test_annotations}')\n",
                "print(f'Test images: {test_images} ({len(list(test_images.glob(\"*.jpg\")))} JPG files)')\n",
                "!python -u src/evaluate_cardd_detector_per_class.py"
            ]
        }

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

colab_setup.ipynb	     patch_colab_notebook.py  src
models			     Project_Specs.md	      train_detector_log.txt
out			     readme.md		      write_colab_ipynb.py
patch_colab_eval.py	     reports
patch_colab_notebook_fix.py  requirments.txt
baseline_model.py		evaluate_cardd_thresholded.py
cardd_audit.py			evaluate_resnet_finetuned.py
cardd_class_weights.py		evaluate_resnet.py
cardd_dataloaders.py		import_cardd.py
cardd_dataset.py		optimize_cardd_thresholds.py
cardd_detection_dataset.py	resnet_model.py
cardd_detector.py		split_dataset.py
cardd_error_analysis.py		test_dataset.py
cardd_gradcam.py		test_detection_training_batch.py
cardd_model.py			train_baseline.py
cardd_visualize_annotations.py	train_cardd_detector.py
data_audit.py			train_cardd_finetune.py
dataloaders.py			train_cardd.py
dataset.py			train_resnet_finetune.py
download_dataset.py		train_resnet.py
evaluate_baseline.py		transforms.py
evaluate_cardd_detector.py	visualize_dataset.py
evaluate_cardd_finetuned.py	visualize_detectio

In [8]:
from pathlib import Path

CARDD_DRIVE = Path(
    "/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO"
)

print("CarDD exists:", CARDD_DRIVE.exists())

print(
    "Train annotations:",
    (CARDD_DRIVE / "annotations" / "instances_train2017.json").exists()
)

print(
    "Validation annotations:",
    (CARDD_DRIVE / "annotations" / "instances_val2017.json").exists()
)

print(
    "Train images:",
    (CARDD_DRIVE / "train2017").exists()
)

CarDD exists: True
Train annotations: True
Validation annotations: True
Train images: True


In [4]:
!mkdir -p "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release"

!rm -rf "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"

!ln -s \
"/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO" \
"/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"

In [5]:
!ls "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO/annotations"

image_info.xlsx		 instances_train2017.json
instances_test2017.json  instances_val2017.json


In [11]:
from pathlib import Path

root = Path(
    "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"
)

print("Train images:", len(list((root / "train2017").glob("*.jpg"))))
print("Val images:", len(list((root / "val2017").glob("*.jpg"))))
print("Test images:", len(list((root / "test2017").glob("*.jpg"))))

Train images: 2816
Val images: 810
Test images: 374


In [12]:
from pathlib import Path

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/vehicle-damage-triage-models"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Permanent checkpoint directory:")
print(CHECKPOINT_DIR)

print("\nExisting files:")
for file in CHECKPOINT_DIR.glob("*"):
    print(file.name)

Permanent checkpoint directory:
/content/drive/MyDrive/vehicle-damage-triage-models

Existing files:


In [13]:
!grep -n "vehicle-damage-triage-models" src/train_cardd_detector.py

47:    "/content/drive/MyDrive/vehicle-damage-triage-models"


In [14]:
!grep -n "optimizer_state_dict" src/train_cardd_detector.py

131:            "optimizer_state_dict": optimizer.state_dict(),


In [ ]:
import torch
from pathlib import Path

print("Working directory:")
        {
            "cell_type": "code",
            "id": "7a8da5e7",
            "metadata": {
                "language": "python"
            },
            "outputs": [],
            "source": [
                "from pathlib import Path\n",
                "import torch\n",
                "from torch.utils.data import DataLoader\n",
                "from torchvision.ops import box_iou\n",
                "from cardd_detection_dataset import CarDDDetectionDataset\n",
                "from cardd_detector import build_detector\n",
                "\n",
                "%cd /content/vehicle-damage-triage\n",
                "checkpoint = Path('/content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth')\n",
                "root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')\n",
                "annotations = root / 'annotations' / 'instances_test2017.json'\n",
                "images = root / 'test2017'\n",
                "for name, path in {'checkpoint': checkpoint, 'annotations': annotations, 'images': images}.items():\n",
                "    if not path.exists():\n",
                "        raise FileNotFoundError(f'{name} not found: {path}')\n",
                "\n",
                "names = {1: 'dent', 2: 'scratch', 3: 'crack', 4: 'glass shatter', 5: 'lamp broken', 6: 'tire flat'}\n",
                "model = build_detector().to('cuda' if torch.cuda.is_available() else 'cpu')\n",
                "device = next(model.parameters()).device\n",
                "model.load_state_dict(torch.load(checkpoint, map_location=device))\n",
                "model.eval()\n",
                "dataset = CarDDDetectionDataset(str(annotations), str(images))\n",
                "loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda batch: tuple(zip(*batch)))\n",
                "counts = {class_id: {'tp': 0, 'fp': 0, 'fn': 0} for class_id in names}\n",
                "\n",
                "with torch.no_grad():\n",
                "    for batch_images, batch_targets in loader:\n",
                "        output = model([batch_images[0].to(device)])[0]\n",
                "        keep = output['scores'].cpu() >= 0.05\n",
                "        pred_boxes = output['boxes'].cpu()[keep]\n",
                "        pred_labels = output['labels'].cpu()[keep]\n",
                "        pred_scores = output['scores'].cpu()[keep]\n",
                "        true_boxes = batch_targets[0]['boxes']\n",
                "        true_labels = batch_targets[0]['labels']\n",
                "        matched = set()\n",
                "        ious = box_iou(pred_boxes, true_boxes) if len(pred_boxes) and len(true_boxes) else None\n",
                "        for pred_index in torch.argsort(pred_scores, descending=True).tolist():\n",
                "            label = int(pred_labels[pred_index])\n",
                "            candidates = [i for i in range(len(true_boxes)) if int(true_labels[i]) == label and i not in matched]\n",
                "            best = max(candidates, key=lambda i: float(ious[pred_index, i])) if candidates and ious is not None else None\n",
                "            if best is not None and float(ious[pred_index, best]) >= 0.5:\n",
                "                matched.add(best); counts[label]['tp'] += 1\n",
                "            elif label in counts:\n",
                "                counts[label]['fp'] += 1\n",
                "        for true_index, label in enumerate(true_labels.tolist()):\n",
                "            if true_index not in matched: counts[int(label)]['fn'] += 1\n",
                "\n",
                "lines = ['Detector error analysis on untouched test2017', '', 'class              TP      FP      FN     precision   recall', '-' * 65]\n",
                "for class_id, name in names.items():\n",
                "    value = counts[class_id]\n",
                "    precision = value['tp'] / (value['tp'] + value['fp']) if value['tp'] + value['fp'] else 0.0\n",
                "    recall = value['tp'] / (value['tp'] + value['fn']) if value['tp'] + value['fn'] else 0.0\n",
                "    lines.append(f'{name:18} {value[\"tp\"]:6d} {value[\"fp\"]:7d} {value[\"fn\"]:7d} {precision:11.4f} {recall:8.4f}')\n",
                "report = Path('reports/cardd_detector_error_analysis.txt')\n",
                "report.parent.mkdir(parents=True, exist_ok=True)\n",
                "report.write_text('\\n'.join(lines) + '\\n', encoding='utf-8')\n",
                "print('\\n'.join(lines))\n",
                "print(f'\\nReport saved to: {report}')"
            ]
        },
                "files.upload(target_dir='/content/vehicle-damage-triage/src')"

print("\nCheckpoint destination:")
checkpoint_root = Path(
    "/content/drive/MyDrive/vehicle-damage-triage-models"
)
print(checkpoint_root)
print("Exists:", checkpoint_root.exists())

Working directory:
/content/vehicle-damage-triage

GPU:
Tesla T4

Dataset:
/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO
Exists: True

Checkpoint destination:
/content/drive/MyDrive/vehicle-damage-triage-models
Exists: True


In [ ]:
!python -u src/train_cardd_detector.py --epochs 5 --batch-size 2

Using device: cuda
Checkpoints will be saved to: /content/drive/MyDrive/vehicle-damage-triage-models
GPU: Tesla T4
Training images: 2816
Validation images: 810
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100% 160M/160M [00:01<00:00, 132MB/s] 

Epoch 1/5
total_loss=0.3847
classifier_loss=0.1721
box_reg_loss=0.1484
objectness_loss=0.0376
rpn_box_loss=0.0266
Saved model permanently to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch1.pth
Saved training state to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch1_training.pth
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.19s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.04s).
Accumulating evaluation results...
DONE (t=0.55s).


In [6]:
!ls -lh "/content/drive/MyDrive/vehicle-damage-triage-models"

total 630M
-rw------- 1 root root 159M Aug 11 12:28 cardd_detector_epoch1.pth
-rw------- 1 root root 157M Aug 11 12:28 cardd_detector_epoch1_training.pth
-rw------- 1 root root 159M Aug 11 12:54 cardd_detector_epoch2.pth
-rw------- 1 root root 157M Aug 11 12:54 cardd_detector_epoch2_training.pth


In [42]:
from pathlib import Path

path = Path('src/train_cardd_detector.py')
text = path.read_text(encoding='utf-8')

if '--resume-epoch' not in text:
    text = text.replace(
        '    return parser.parse_args()',
        '''    parser.add_argument(
        "--resume-epoch",
        type=int,
        default=None,
        help="Resume from the specified saved epoch.",
    )

    return parser.parse_args()''',
        1,
    )

if 'start_epoch = 1' not in text:
    optimizer_marker = '''    optimizer = torch.optim.SGD(
        trainable_parameters,
        lr=0.005,
        momentum=0.9,
        weight_decay=0.0005,
    )'''
    resume_block = optimizer_marker + '''

    start_epoch = 1
    if args.resume_epoch is not None:
        resume_model_path = MODELS_DIR / f"cardd_detector_epoch{args.resume_epoch}.pth"
        resume_training_path = MODELS_DIR / f"cardd_detector_epoch{args.resume_epoch}_training.pth"
        if not resume_model_path.exists():
            raise FileNotFoundError(f"Model checkpoint not found: {resume_model_path}")
        if not resume_training_path.exists():
            raise FileNotFoundError(f"Training checkpoint not found: {resume_training_path}")

        print(f"Loading model checkpoint: {resume_model_path}")
        model.load_state_dict(torch.load(resume_model_path, map_location=device))
        training_checkpoint = torch.load(resume_training_path, map_location=device)
        optimizer.load_state_dict(training_checkpoint["optimizer_state_dict"])
        start_epoch = args.resume_epoch + 1
        print(f"Resuming after epoch {args.resume_epoch}. Next epoch: {start_epoch}")'''
    if optimizer_marker not in text:
        raise RuntimeError('Could not find optimizer block in trainer.')
    text = text.replace(optimizer_marker, resume_block, 1)

text = text.replace(
    '''    for epoch in range(
        1,
        args.epochs + 1,
    ):''',
    '''    for epoch in range(
        start_epoch,
        args.epochs + 1,
    ):''',
    1,
)
path.write_text(text, encoding='utf-8')
print('Trainer resume support is ready.')

Trainer resume support is ready.


In [8]:
!grep -n "resume-epoch" src/train_cardd_detector.py
!grep -n "start_epoch" src/train_cardd_detector.py

310:        "--resume-epoch",
447:    start_epoch = 1
491:        start_epoch = resume_epoch + 1
495:            f"Next epoch: {start_epoch}"
503:        start_epoch,


In [9]:
!python -u src/train_cardd_detector.py \
    --epochs 5 \
    --batch-size 2 \
    --resume-epoch 2

Using device: cuda
Checkpoints will be saved to: /content/drive/MyDrive/vehicle-damage-triage-models
GPU: Tesla T4
Training images: 2816
Validation images: 810
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100% 160M/160M [00:00<00:00, 190MB/s] 
Loading model checkpoint: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch2.pth
Resuming after epoch 2. Next epoch: 3

Epoch 3/5
total_loss=0.3023
classifier_loss=0.1293
box_reg_loss=0.1368
objectness_loss=0.0155
rpn_box_loss=0.0207
Saved model permanently to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth
Saved training state to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3_training.pth
loading annotations into memory...
Done (t=0.12s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
inde

In [38]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!git clone https://github.com/AymanLakhnati/AI-vehicule-damage-triage vehicle-damage-triage
%cd /content/vehicle-damage-triage

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'vehicle-damage-triage' already exists and is not an empty directory.
/content/vehicle-damage-triage


In [39]:
!pip install -q torch torchvision torchmetrics pycocotools scipy

In [ ]:
from pathlib import Path

# Write the test evaluator script
test_evaluator_code = """from pathlib import Path

import torch
from torch.utils.data import DataLoader

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except ImportError as exc:
    raise ImportError(
        'torchmetrics is required for evaluation. Install it with: py -m pip install torchmetrics'
    ) from exc

from cardd_detection_dataset import CarDDDetectionDataset
from cardd_detector import build_detector
                "from pathlib import Path\n",
                "\n",
                "checkpoint = Path('/content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth')\n",
                "dataset_root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')\n",
                "annotations = dataset_root / 'annotations' / 'instances_test2017.json'\n",
                "images = dataset_root / 'test2017'\n",
                "\n",
                "for name, path in {'checkpoint': checkpoint, 'annotations': annotations, 'images': images}.items():\n",
                "    if not path.exists():\n",
                "        raise FileNotFoundError(f'{name} not found: {path}')\n",
                "\n",
                "!python -u src/analyze_cardd_detector_errors.py \\\n",
                "    --checkpoint $checkpoint \\\n",
                "    --annotations $annotations \\\n",
                "    --images $images"
DATA_ROOT = REPO_ROOT / 'data' / 'raw' / 'cardd' / 'CarDD_release' / 'CarDD_COCO'
MODELS_DIR = REPO_ROOT / 'models'

# Test-specific paths
TEST_ANNOTATIONS_PATH = (DATA_ROOT / 'annotations' / 'instances_test2017.json')
TEST_IMAGES_DIR = DATA_ROOT / 'test2017'
CHECKPOINT_PATH = Path(
    '/content/drive/MyDrive/vehicle-damage-triage-models/'
    'cardd_detector_epoch3.pth'
)

REPORT_PATH = REPO_ROOT / 'reports' / 'cardd_detector_test_results.txt'
BATCH_SIZE = 2
NUM_WORKERS = 0


def collate_fn(batch):
    return tuple(zip(*batch))


def load_model(checkpoint_path: Path, device: torch.device) -> torch.nn.Module:
    model = build_detector().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def build_test_loader() -> DataLoader:
    dataset = CarDDDetectionDataset(str(TEST_ANNOTATIONS_PATH), str(TEST_IMAGES_DIR))
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
    )


def evaluate(model: torch.nn.Module, data_loader: DataLoader, device: torch.device) -> dict:
    metric = MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        class_metrics=True,
    )

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            predictions = [
                {
                    'boxes': out['boxes'].cpu(),
                    'scores': out['scores'].cpu(),
                    'labels': out['labels'].cpu(),
                }
                for out in outputs
            ]
            reference_targets = [
                {
                    'boxes': tgt['boxes'].cpu(),
                    'labels': tgt['labels'].cpu(),
                }
                for tgt in targets
            ]

            metric.update(predictions, reference_targets)

    results = metric.compute()
    return {
        'mAP': results['map'].item(),
        'mAP_50': results['map_50'].item(),
        'mAP_75': results['map_75'].item(),
        'mAR_100': results['mar_100'].item(),
        'per_class_AP': results.get('map_per_class'),
        'per_class_AR': results.get('mar_100_per_class'),
    }


def save_results(results: dict) -> None:
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    lines = [
        f'Test mAP@[0.50:0.95]: {results["mAP"]:.6f}',
        f'Test mAP@50: {results["mAP_50"]:.6f}',
        f'Test mAP@75: {results["mAP_75"]:.6f}',
        f'Test AR@100: {results["mAR_100"]:.6f}',
        '',
    ]

    if results['per_class_AP'] is not None:
        lines.append('per_class_AP:')
        for idx, value in enumerate(results['per_class_AP']):
            lines.append(f'  class_{idx + 1}: {value:.6f}')
        lines.append('')

    if results['per_class_AR'] is not None:
        lines.append('per_class_AR@100:')
        for idx, value in enumerate(results['per_class_AR']):
            lines.append(f'  class_{idx + 1}: {value:.6f}')
        lines.append('')

    REPORT_PATH.write_text('\\n'.join(lines) + '\\n', encoding='utf-8')


def main() -> None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    print(f'Loading checkpoint: {CHECKPOINT_PATH}')
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')

    model = load_model(CHECKPOINT_PATH, device)
    test_loader = build_test_loader()

    results = evaluate(model, test_loader, device)

    print(f'Test mAP@[0.50:0.95]: {results["mAP"]:.6f}')
    print(f'Test mAP@50: {results["mAP_50"]:.6f}')
    print(f'Test mAP@75: {results["mAP_75"]:.6f}')
    print(f'Test AR@100: {results["mAR_100"]:.6f}')

    save_results(results)
    print(f'Results saved to: {REPORT_PATH}')


if __name__ == '__main__':
    main()
"""

script_path = Path('/content/vehicle-damage-triage/src/evaluate_cardd_detector_test.py')
script_path.write_text(test_evaluator_code, encoding='utf-8')
print('Test evaluator script created successfully.')

Test evaluator script created successfully.


In [40]:
# Diagnose CarDD dataset location
from pathlib import Path
import os

# Check various possible Drive paths
possible_paths = [
    Path("/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO"),
    Path("/content/drive/MyDrive/CarDD_COCO"),
    Path("/content/drive/MyDrive/CarDD_release/CarDD_COCO"),
]

found_path = None
print("Checking for CarDD dataset in Drive:")
for path in possible_paths:
    exists = path.exists()
    print(f"  {path}: {exists}")
    if exists:
        found_path = path

if not found_path:
    print("\n⚠️  CarDD dataset not found in expected Drive locations!")
    print("Please check:")
    print("  1. Is the CarDD dataset uploaded to your Google Drive?")
    print("  2. Is it in the correct folder path?")
    print("\nActual Drive root contents:")
    drive_root = Path("/content/drive/MyDrive")
    for item in sorted(drive_root.iterdir())[:20]:
        print(f"    {item.name}")
else:
    print(f"\n✓ Found CarDD dataset at: {found_path}")
    print(f"Contents:")
    for item in sorted(found_path.iterdir()):
        print(f"  {item.name}")
    
    # Create symlink using shell commands (more reliable in Colab)
    print("\nCreating symlink...")
    !mkdir -p "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release"
    !rm -rf "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"
    !ln -s "/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO" "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"
    print("✓ Symlink created successfully")

Checking for CarDD dataset in Drive:
  /content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO: True
  /content/drive/MyDrive/CarDD_COCO: False
  /content/drive/MyDrive/CarDD_release/CarDD_COCO: False

✓ Found CarDD dataset at: /content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO
Contents:
  annotations
  test2017
  train2017
  val2017

Creating symlink...
✓ Symlink created successfully


In [41]:
# Check what files actually exist
from pathlib import Path

cardd_root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')
annotations_dir = cardd_root / 'annotations'

print("Available annotation files:")
for file in sorted(annotations_dir.glob('*.json')):
    print(f"  {file.name}")

print("\nAvailable image directories:")
for d in sorted(cardd_root.iterdir()):
    if d.is_dir() and d.name != 'annotations':
        img_count = len(list(d.glob('*.jpg')))
        print(f"  {d.name}: {img_count} images")

Available annotation files:
  instances_test2017.json
  instances_train2017.json
  instances_val2017.json

Available image directories:
  test2017: 374 images
  train2017: 2816 images
  val2017: 810 images


In [11]:
!python -u src/evaluate_cardd_detector_test.py

Using device: cuda
Loading checkpoint: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth
Test mAP@[0.50:0.95]: 0.392843
Test mAP@50: 0.599369
Test mAP@75: 0.411063
Test AR@100: 0.566247
Results saved to: /content/vehicle-damage-triage/reports/cardd_detector_test_results.txt


In [13]:
from pathlib import Path

# Write the prediction script
predict_code = """import argparse
from pathlib import Path

import torch
import torchvision.transforms as transforms
from PIL import Image, ImageDraw, ImageFont

from cardd_detector import build_detector

# Class names for vehicle damage detection
CLASS_NAMES = {
    1: "dent",
    2: "scratch",
    3: "crack",
    4: "glass shatter",
    5: "lamp broken",
    6: "tire flat",
}

# Paths
REPO_ROOT = Path(__file__).resolve().parents[1]
CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/vehicle-damage-triage-models/"
    "cardd_detector_epoch3.pth"
)
OUTPUT_DIR = REPO_ROOT / "out"
CONFIDENCE_THRESHOLD = 0.50


def load_model(checkpoint_path: Path, device: torch.device) -> torch.nn.Module:
    model = build_detector().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def load_and_preprocess_image(image_path: Path, device: torch.device) -> tuple:
    image = Image.open(image_path).convert("RGB")
    image_tensor = transforms.ToTensor()(image).to(device)
    return image, image_tensor


def predict_single_image(
    image_path: Path,
    model: torch.nn.Module,
    device: torch.device,
    confidence_threshold: float = 0.50,
) -> dict:
    image, image_tensor = load_and_preprocess_image(image_path, device)

    with torch.no_grad():
        outputs = model([image_tensor])

    predictions = {
        "boxes": outputs[0]["boxes"].cpu().numpy(),
        "labels": outputs[0]["labels"].cpu().numpy(),
        "scores": outputs[0]["scores"].cpu().numpy(),
    }

    mask = predictions["scores"] >= confidence_threshold
    predictions["boxes"] = predictions["boxes"][mask]
    predictions["labels"] = predictions["labels"][mask]
    predictions["scores"] = predictions["scores"][mask]

    return image, predictions


def draw_predictions(
    image: Image.Image,
    predictions: dict,
    class_names: dict,
    output_path: Path,
) -> None:
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 16)
        font_small = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12
        )
    except (IOError, OSError):
        font = ImageFont.load_default()
        font_small = font

    colors = {
        1: (255, 0, 0),
        2: (0, 255, 0),
        3: (0, 0, 255),
        4: (255, 255, 0),
        5: (255, 0, 255),
        6: (0, 255, 255),
    }

    boxes = predictions["boxes"]
    labels = predictions["labels"]
    scores = predictions["scores"]

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        label_name = class_names.get(int(label), f"class_{label}")
        color = colors.get(int(label), (255, 255, 255))

        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

        label_text = f"{label_name}: {score:.2f}"
        bbox = draw.textbbox((x1, y1 - 20), label_text, font=font)
        draw.rectangle(bbox, fill=color)
        draw.text((x1, y1 - 20), label_text, fill=(0, 0, 0), font=font)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path)


def main():
    parser = argparse.ArgumentParser(
        description="Predict vehicle damage on a single image using Epoch 3 detector."
    )
    parser.add_argument(
        "--image",
        type=str,
        required=True,
        help="Path to the input car image.",
    )
    parser.add_argument(
        "--output",
        type=str,
        default="out/detection_prediction.jpg",
        help="Path to save the annotated output image (default: out/detection_prediction.jpg).",
    )
    parser.add_argument(
        "--confidence",
        type=float,
        default=0.50,
        help="Confidence threshold for detections (default: 0.50).",
    )

    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print(f"Loading checkpoint: {CHECKPOINT_PATH}")
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

    model = load_model(CHECKPOINT_PATH, device)

    image_path = Path(args.image)
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    print(f"Processing image: {image_path}")
    image, predictions = predict_single_image(
        image_path, model, device, confidence_threshold=args.confidence
    )

    print("\\nDetected damages:")
    class_scores = {}
    for label, score in zip(predictions["labels"], predictions["scores"]):
        label_name = CLASS_NAMES.get(int(label), f"class_{label}")
        if label_name not in class_scores:
            class_scores[label_name] = []
        class_scores[label_name].append(score)

    for class_name in sorted(class_scores.keys()):
        avg_score = sum(class_scores[class_name]) / len(class_scores[class_name])
        count = len(class_scores[class_name])
        print(f"  - {class_name}: {avg_score:.2f} (detected {count} times)")

    if not predictions["labels"].size:
        print("  (No damage detected above confidence threshold)")

    output_path = Path(args.output)
    draw_predictions(image, predictions, CLASS_NAMES, output_path)
    print(f"\\nSaved prediction to: {output_path}")


if __name__ == "__main__":
    main()
"""

script_path = Path('/content/vehicle-damage-triage/src/predict_cardd_detector.py')
script_path.write_text(predict_code, encoding='utf-8')
print('Prediction script created successfully.')

Prediction script created successfully.


In [15]:
# Find first test image
from pathlib import Path

test_dir = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO/test2017')
test_images = sorted(test_dir.glob('*.jpg'))

if test_images:
    first_image = test_images[0]
    print(f"Found {len(test_images)} test images")
    print(f"Using first image: {first_image.name}")
else:
    print("No test images found!")

Found 374 test images
Using first image: 000012.jpg


In [16]:
from pathlib import Path

# Find and use first test image
test_dir = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO/test2017')
test_images = sorted(test_dir.glob('*.jpg'))

if test_images:
    first_image = test_images[0]
    !python src/predict_cardd_detector.py --image {first_image} --output out/detection_example.jpg
else:
    print("No test images found!")

Using device: cuda
Loading checkpoint: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth
Processing image: /content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO/test2017/000012.jpg

Detected damages:
  - tire flat: 0.95 (detected 1 times)

Saved prediction to: out/detection_example.jpg


In [5]:
from pathlib import Path

# Write the per-class evaluation script
per_class_eval_code = """from pathlib import Path

import torch
from torch.utils.data import DataLoader

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except ImportError as exc:
    raise ImportError(
        'torchmetrics is required for evaluation. Install it with: py -m pip install torchmetrics'
    ) from exc

from cardd_detection_dataset import CarDDDetectionDataset
from cardd_detector import build_detector

# Class names for vehicle damage detection
CLASS_NAMES = {
    1: "dent",
    2: "scratch",
    3: "crack",
    4: "glass shatter",
    5: "lamp broken",
    6: "tire flat",
}

REPO_ROOT = Path(__file__).resolve().parents[1]
DATA_ROOT = REPO_ROOT / 'data' / 'raw' / 'cardd' / 'CarDD_release' / 'CarDD_COCO'
MODELS_DIR = REPO_ROOT / 'models'

# Test-specific paths
TEST_ANNOTATIONS_PATH = (DATA_ROOT / 'annotations' / 'instances_test2017.json')
TEST_IMAGES_DIR = DATA_ROOT / 'test2017'
CHECKPOINT_PATH = Path(
    '/content/drive/MyDrive/vehicle-damage-triage-models/'
    'cardd_detector_epoch3.pth'
)

REPORT_PATH = REPO_ROOT / 'reports' / 'cardd_detector_per_class_results.txt'
BATCH_SIZE = 2
NUM_WORKERS = 0


def collate_fn(batch):
    return tuple(zip(*batch))


def load_model(checkpoint_path: Path, device: torch.device) -> torch.nn.Module:
    model = build_detector().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def build_test_loader() -> DataLoader:
    dataset = CarDDDetectionDataset(str(TEST_ANNOTATIONS_PATH), str(TEST_IMAGES_DIR))
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
    )


def evaluate_per_class(
    model: torch.nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> dict:
    metric = MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        class_metrics=True,
    )

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            predictions = [
                {
                    'boxes': out['boxes'].cpu(),
                    'scores': out['scores'].cpu(),
                    'labels': out['labels'].cpu(),
                }
                for out in outputs
            ]
            reference_targets = [
                {
                    'boxes': tgt['boxes'].cpu(),
                    'labels': tgt['labels'].cpu(),
                }
                for tgt in targets
            ]

            metric.update(predictions, reference_targets)

    results = metric.compute()

    per_class_ap = results.get('map_per_class')
    
    return {
        'mAP': results['map'].item(),
        'mAP_50': results['map_50'].item(),
        'mAP_75': results['map_75'].item(),
        'mAR_100': results['mar_100'].item(),
        'per_class_AP': per_class_ap,
    }


def save_results(results: dict, class_names: dict) -> None:
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    lines = [
        'Per-class test performance (Epoch 3)',
        '=' * 50,
        '',
    ]

    lines.append(f'Overall mAP@[0.50:0.95]: {results["mAP"]:.6f}')
    lines.append(f'Overall mAP@50: {results["mAP_50"]:.6f}')
    lines.append(f'Overall mAP@75: {results["mAP_75"]:.6f}')
    lines.append(f'Overall mAR@100: {results["mAR_100"]:.6f}')
    lines.append('')
    lines.append('Per-class AP:')
    lines.append('-' * 50)

    per_class_ap = results['per_class_AP']
    if per_class_ap is not None:
        for class_id in sorted(class_names.keys()):
            class_name = class_names[class_id]
            ap_value = per_class_ap[class_id - 1].item()
            lines.append(f'{class_name:18} AP = {ap_value:.6f}')
    else:
        lines.append('(Per-class metrics not available)')

    REPORT_PATH.write_text('\\n'.join(lines) + '\\n', encoding='utf-8')


def main() -> None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    print(f'Loading checkpoint: {CHECKPOINT_PATH}')
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')

    model = load_model(CHECKPOINT_PATH, device)
    test_loader = build_test_loader()

    print('Evaluating on test set...')
    results = evaluate_per_class(model, test_loader, device)

    print('\\nPer-class test performance')
    print('=' * 50)
    print(f'Overall mAP@[0.50:0.95]: {results["mAP"]:.6f}')
    print(f'Overall mAP@50: {results["mAP_50"]:.6f}')
    print(f'Overall mAP@75: {results["mAP_75"]:.6f}')
    print(f'Overall mAR@100: {results["mAR_100"]:.6f}')
    print('')
    print('Per-class AP:')
    print('-' * 50)

    per_class_ap = results['per_class_AP']
    if per_class_ap is not None:
        for class_id in sorted(CLASS_NAMES.keys()):
            class_name = CLASS_NAMES[class_id]
            ap_value = per_class_ap[class_id - 1].item()
            print(f'{class_name:18} AP = {ap_value:.6f}')
    else:
        print('(Per-class metrics not available)')

    save_results(results, CLASS_NAMES)
    print(f'\\nResults saved to: {REPORT_PATH}')


if __name__ == '__main__':
    main()
"""

script_path = Path('/content/vehicle-damage-triage/src/evaluate_cardd_detector_per_class.py')
script_path.write_text(per_class_eval_code, encoding='utf-8')
print('Per-class evaluator script created successfully.')

Per-class evaluator script created successfully.


In [6]:
!python src/evaluate_cardd_detector_per_class.py

Using device: cuda
Loading checkpoint: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100% 160M/160M [00:00<00:00, 203MB/s] 
Evaluating on test set...

Per-class test performance
Overall mAP@[0.50:0.95]: 0.392843
Overall mAP@50: 0.599369
Overall mAP@75: 0.411063
Overall mAR@100: 0.566247

Per-class AP:
--------------------------------------------------
dent               AP = 0.171560
scratch            AP = 0.211990
crack              AP = 0.141393
glass shatter      AP = 0.755795
lamp broken        AP = 0.398128
tire flat          AP = 0.678189

Results saved to: /content/vehicle-damage-triage/reports/cardd_detector_per_class_results.txt


In [35]:
%cd /content/vehicle-damage-triage

import sys
from pathlib import Path

sys.path.insert(0, '/content/vehicle-damage-triage/src')

import torch
from torch.utils.data import DataLoader
from torchvision.ops import box_iou
from cardd_detection_dataset import CarDDDetectionDataset
from cardd_detector import build_detector

checkpoint = Path('/content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth')
root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')
annotations = root / 'annotations' / 'instances_test2017.json'
images = root / 'test2017'
required = {'checkpoint': checkpoint, 'annotations': annotations, 'images': images}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

names = {1: 'dent', 2: 'scratch', 3: 'crack', 4: 'glass shatter', 5: 'lamp broken', 6: 'tire flat'}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model = build_detector().to(device)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()
dataset = CarDDDetectionDataset(str(annotations), str(images))
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda batch: tuple(zip(*batch)))
counts = {class_id: {'tp': 0, 'fp': 0, 'fn': 0} for class_id in names}

with torch.no_grad():
    for batch_images, batch_targets in loader:
        output = model([batch_images[0].to(device)])[0]
        keep = output['scores'].cpu() >= 0.05
        pred_boxes = output['boxes'].cpu()[keep]
        pred_labels = output['labels'].cpu()[keep]
        pred_scores = output['scores'].cpu()[keep]
        true_boxes = batch_targets[0]['boxes']
        true_labels = batch_targets[0]['labels']
        matched = set()
        ious = box_iou(pred_boxes, true_boxes) if len(pred_boxes) and len(true_boxes) else None
        for pred_index in torch.argsort(pred_scores, descending=True).tolist():
            label = int(pred_labels[pred_index])
            candidates = [i for i in range(len(true_boxes)) if int(true_labels[i]) == label and i not in matched]
            best = max(candidates, key=lambda i: float(ious[pred_index, i])) if candidates and ious is not None else None
            if best is not None and float(ious[pred_index, best]) >= 0.5:
                matched.add(best)
                counts[label]['tp'] += 1
            elif label in counts:
                counts[label]['fp'] += 1
        for true_index, label in enumerate(true_labels.tolist()):
            if true_index not in matched:
                counts[int(label)]['fn'] += 1

lines = ['Detector error analysis on untouched test2017', '', 'class              TP      FP      FN     precision   recall', '-' * 65]
for class_id, name in names.items():
    value = counts[class_id]
    precision = value['tp'] / (value['tp'] + value['fp']) if value['tp'] + value['fp'] else 0.0
    recall = value['tp'] / (value['tp'] + value['fn']) if value['tp'] + value['fn'] else 0.0
    lines.append(f'{name:18} {value["tp"]:6d} {value["fp"]:7d} {value["fn"]:7d} {precision:11.4f} {recall:8.4f}')
report = Path('reports/cardd_detector_error_analysis.txt')
report.parent.mkdir(parents=True, exist_ok=True)
report.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines))
print(f'\nReport saved to: {report}')

/content/vehicle-damage-triage
Using device: cuda
Detector error analysis on untouched test2017

class              TP      FP      FN     precision   recall
-----------------------------------------------------------------
dent                  181    1485      55      0.1086   0.7669
scratch               265    3035      42      0.0803   0.8632
crack                  52     907      18      0.0542   0.7429
glass shatter          70     208       1      0.2518   0.9859
lamp broken            66     415       3      0.1372   0.9565
tire flat              30     228       2      0.1163   0.9375

Report saved to: reports/cardd_detector_error_analysis.txt


In [7]:
%cd /content/vehicle-damage-triage

import json
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision.ops import box_iou

sys.path.insert(0, '/content/vehicle-damage-triage/src')
from cardd_detection_dataset import CarDDDetectionDataset
from cardd_detector import build_detector

checkpoint = Path('/content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch4.pth')
root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')
annotations = root / 'annotations' / 'instances_val2017.json'
images = root / 'val2017'
required = {'checkpoint': checkpoint, 'validation annotations': annotations, 'validation images': images}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

names = {1: 'dent', 2: 'scratch', 3: 'crack', 4: 'glass shatter', 5: 'lamp broken', 6: 'tire flat'}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model = build_detector().to(device)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()
dataset = CarDDDetectionDataset(str(annotations), str(images))
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0, collate_fn=lambda batch: tuple(zip(*batch)))
all_predictions = []
with torch.no_grad():
    for batch_images, batch_targets in loader:
        outputs = model([image.to(device) for image in batch_images])
        for output, target in zip(outputs, batch_targets):
            all_predictions.append(({key: value.cpu() for key, value in output.items()}, target))

thresholds = [round(value / 100, 2) for value in range(5, 100, 5)]
def score(class_id, threshold):
    tp = fp = fn = 0
    for prediction, target in all_predictions:
        keep = prediction['scores'] >= threshold
        boxes = prediction['boxes'][keep]
        labels = prediction['labels'][keep]
        truths = target['boxes']
        true_labels = target['labels']
        matched = set()
        ious = box_iou(boxes, truths) if len(boxes) and len(truths) else None
        for index in torch.argsort(prediction['scores'][keep], descending=True).tolist():
            if int(labels[index]) != class_id:
                continue
            candidates = [i for i in range(len(truths)) if int(true_labels[i]) == class_id and i not in matched]
            best = max(candidates, key=lambda i: float(ious[index, i])) if candidates and ious is not None else None
            if best is not None and float(ious[index, best]) >= 0.5:
                matched.add(best)
                tp += 1
            else:
                fp += 1
        fn += sum(1 for i, label in enumerate(true_labels.tolist()) if int(label) == class_id and i not in matched)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'threshold': threshold, 'precision': precision, 'recall': recall, 'f1': f1}

selected = {}
for class_id, name in names.items():
    best = max((score(class_id, threshold) for threshold in thresholds), key=lambda result: (result['f1'], result['recall'], -result['threshold']))
    selected[name] = best['threshold']
    print(f'{name:18} threshold={best["threshold"]:.2f} precision={best["precision"]:.4f} recall={best["recall"]:.4f} F1={best["f1"]:.4f}')

output = Path('models/cardd_detector_thresholds_epoch4.json')
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(selected, indent=2) + '\n', encoding='utf-8')
print(f'Saved validation-tuned thresholds to: {output}')

/content/vehicle-damage-triage
Using device: cuda
dent               threshold=0.55 precision=0.4420 recall=0.4032 F1=0.4217
scratch            threshold=0.65 precision=0.5385 recall=0.4519 F1=0.4914
crack              threshold=0.30 precision=0.4854 recall=0.2825 F1=0.3571
glass shatter      threshold=0.65 precision=0.9773 recall=0.9556 F1=0.9663
lamp broken        threshold=0.80 precision=0.8476 recall=0.6312 F1=0.7236
tire flat          threshold=0.80 precision=0.9574 recall=0.7258 F1=0.8257
Saved validation-tuned thresholds to: models/cardd_detector_thresholds_epoch4.json


In [8]:
%cd /content/vehicle-damage-triage

import json
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision.ops import box_iou

sys.path.insert(0, '/content/vehicle-damage-triage/src')
from cardd_detection_dataset import CarDDDetectionDataset
from cardd_detector import build_detector

checkpoint = Path('/content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch4.pth')
threshold_file = Path('models/cardd_detector_thresholds_epoch4.json')
root = Path('/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO')
annotations = root / 'annotations' / 'instances_test2017.json'
images = root / 'test2017'
required = {'checkpoint': checkpoint, 'threshold file': threshold_file, 'test annotations': annotations, 'test images': images}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

names = {1: 'dent', 2: 'scratch', 3: 'crack', 4: 'glass shatter', 5: 'lamp broken', 6: 'tire flat'}
thresholds_by_name = json.loads(threshold_file.read_text(encoding='utf-8'))
thresholds = {class_id: float(thresholds_by_name[name]) for class_id, name in names.items()}
print('Using Epoch 4 validation-tuned thresholds:', thresholds_by_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_detector().to(device)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()
dataset = CarDDDetectionDataset(str(annotations), str(images))
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0, collate_fn=lambda batch: tuple(zip(*batch)))
counts = {class_id: {'tp': 0, 'fp': 0, 'fn': 0} for class_id in names}

with torch.no_grad():
    for batch_images, batch_targets in loader:
        outputs = model([image.to(device) for image in batch_images])
        for output, target in zip(outputs, batch_targets):
            scores = output['scores'].cpu()
            labels = output['labels'].cpu()
            boxes = output['boxes'].cpu()
            keep = torch.tensor([scores[i] >= thresholds.get(int(labels[i]), 1.0) for i in range(len(scores))], dtype=torch.bool)
            boxes, labels, scores = boxes[keep], labels[keep], scores[keep]
            true_boxes, true_labels = target['boxes'], target['labels']
            matched = set()
            ious = box_iou(boxes, true_boxes) if len(boxes) and len(true_boxes) else None
            for pred_index in torch.argsort(scores, descending=True).tolist():
                class_id = int(labels[pred_index])
                if class_id not in counts:
                    continue
                candidates = [i for i in range(len(true_boxes)) if int(true_labels[i]) == class_id and i not in matched]
                best = max(candidates, key=lambda i: float(ious[pred_index, i])) if candidates and ious is not None else None
                if best is not None and float(ious[pred_index, best]) >= 0.5:
                    matched.add(best)
                    counts[class_id]['tp'] += 1
                else:
                    counts[class_id]['fp'] += 1
            for true_index, class_id in enumerate(true_labels.tolist()):
                if true_index not in matched and int(class_id) in counts:
                    counts[int(class_id)]['fn'] += 1

lines = ['Final Epoch 4 thresholded detector evaluation on untouched test2017', '', 'class              TP      FP      FN     precision   recall       F1', '-' * 76]
for class_id, name in names.items():
    value = counts[class_id]
    precision = value['tp'] / (value['tp'] + value['fp']) if value['tp'] + value['fp'] else 0.0
    recall = value['tp'] / (value['tp'] + value['fn']) if value['tp'] + value['fn'] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    lines.append(f'{name:18} {value["tp"]:6d} {value["fp"]:7d} {value["fn"]:7d} {precision:11.4f} {recall:8.4f} {f1:9.4f}')
report = Path('reports/cardd_detector_thresholded_epoch4_results.txt')
report.parent.mkdir(parents=True, exist_ok=True)
report.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines))
print(f'\nReport saved to: {report}')

/content/vehicle-damage-triage
Using Epoch 4 validation-tuned thresholds: {'dent': 0.55, 'scratch': 0.65, 'crack': 0.3, 'glass shatter': 0.65, 'lamp broken': 0.8, 'tire flat': 0.8}
Final Epoch 4 thresholded detector evaluation on untouched test2017

class              TP      FP      FN     precision   recall       F1
----------------------------------------------------------------------------
dent                  115     146     121      0.4406   0.4873    0.4628
scratch               145     135     162      0.5179   0.4723    0.4940
crack                  23      26      47      0.4694   0.3286    0.3866
glass shatter          64       3       7      0.9552   0.9014    0.9275
lamp broken            34       7      35      0.8293   0.4928    0.6182
tire flat              26       3       6      0.8966   0.8125    0.8525

Report saved to: reports/cardd_detector_thresholded_epoch4_results.txt


In [43]:
from pathlib import Path

checkpoint_dir = Path('/content/drive/MyDrive/vehicle-damage-triage-models')
epoch3_model = checkpoint_dir / 'cardd_detector_epoch3.pth'
epoch3_training = checkpoint_dir / 'cardd_detector_epoch3_training.pth'

if not epoch3_model.exists() or not epoch3_training.exists():
    raise FileNotFoundError(
        'Epoch 3 model and training-state checkpoints are required before fine-tuning.\n'
        f'Model: {epoch3_model}\nTraining state: {epoch3_training}'
    )

%cd /content/vehicle-damage-triage
print('Fine-tuning from Epoch 3 to Epoch 5 using train2017 and val2017 only.')
print('The untouched test2017 split is not used during training.')
!python -u src/train_cardd_detector.py --epochs 5 --batch-size 2 --resume-epoch 3

/content/vehicle-damage-triage
Fine-tuning from Epoch 3 to Epoch 5 using train2017 and val2017 only.
The untouched test2017 split is not used during training.
Using device: cuda
Checkpoints will be saved to: /content/drive/MyDrive/vehicle-damage-triage-models
GPU: Tesla T4
Training images: 2816
Validation images: 810
Loading model checkpoint: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch3.pth
Resuming after epoch 3. Next epoch: 4

Epoch 4/5
total_loss=0.2822
classifier_loss=0.1195
box_reg_loss=0.1305
objectness_loss=0.0126
rpn_box_loss=0.0196
Saved model permanently to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch4.pth
Saved training state to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch4_training.pth
loading annotations into memory...
Done (t=0.07s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate a

In [ ]:
print('Detector error analysis is performed by Cell 31. No separate script upload or command is required.')

python3: can't open file '/content/vehicle-damage-triage/src/analyze_cardd_detector_errors.py': [Errno 2] No such file or directory


In [6]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

repo_dir = '/content/vehicle-damage-triage'
src_dir = '/content/vehicle-damage-triage/src'
required_source = [
    f'{src_dir}/cardd_detection_dataset.py',
    f'{src_dir}/cardd_detector.py',
]

if not all(Path(path).exists() for path in required_source):
    if Path(repo_dir).exists():
        subprocess.run(['rm', '-rf', repo_dir], check=True)
    subprocess.run([
        'git', 'clone',
        'https://github.com/AymanLakhnati/AI-vehicule-damage-triage',
        repo_dir,
    ], check=True)

if not all(Path(path).exists() for path in required_source):
    raise FileNotFoundError(f'Required project modules are missing from {src_dir}')

def load_module(module_name, module_path):
    specification = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(specification)
    sys.modules[module_name] = module
    specification.loader.exec_module(module)
    return module

dataset_module = load_module('cardd_detection_dataset', f'{src_dir}/cardd_detection_dataset.py')
detector_module = load_module('cardd_detector', f'{src_dir}/cardd_detector.py')
CarDDDetectionDataset = dataset_module.CarDDDetectionDataset
build_detector = detector_module.build_detector

os.chdir(repo_dir)
print(f'Working directory: {os.getcwd()}')
print('Loaded project modules directly from absolute paths.')
!pip install -q torchmetrics pycocotools

import torch
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision

cardd_drive = Path('/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO')
cardd_link = Path(repo_dir) / 'data' / 'raw' / 'cardd' / 'CarDD_release' / 'CarDD_COCO'
if not cardd_link.exists():
    cardd_link.parent.mkdir(parents=True, exist_ok=True)
    cardd_link.symlink_to(cardd_drive, target_is_directory=True)

checkpoint_dir = Path('/content/drive/MyDrive/vehicle-damage-triage-models')
test_annotations = cardd_link / 'annotations' / 'instances_test2017.json'
test_images = cardd_link / 'test2017'
required = {
    'Epoch 4 checkpoint': checkpoint_dir / 'cardd_detector_epoch4.pth',
    'Epoch 5 checkpoint': checkpoint_dir / 'cardd_detector_epoch5.pth',
    'test annotations': test_annotations,
    'test images': test_images,
}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

names = {1: 'dent', 2: 'scratch', 3: 'crack', 4: 'glass shatter', 5: 'lamp broken', 6: 'tire flat'}
def collate_fn(batch):
    return tuple(zip(*batch))

def evaluate(checkpoint):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_detector().to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()
    dataset = CarDDDetectionDataset(str(test_annotations), str(test_images))
    loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0, collate_fn=collate_fn)
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=True)
    with torch.no_grad():
        for images, targets in loader:
            outputs = model([image.to(device) for image in images])
            predictions = [{'boxes': output['boxes'].cpu(), 'scores': output['scores'].cpu(), 'labels': output['labels'].cpu()} for output in outputs]
            references = [{'boxes': target['boxes'].cpu(), 'labels': target['labels'].cpu()} for target in targets]
            metric.update(predictions, references)
    return metric.compute()

results = {}
for epoch in (4, 5):
    checkpoint = checkpoint_dir / f'cardd_detector_epoch{epoch}.pth'
    print(f'\nEvaluating Epoch {epoch} on untouched test2017...')
    result = evaluate(checkpoint)
    results[epoch] = result
    print(f'Overall mAP@[0.50:0.95]: {result["map"].item():.6f}')
    print(f'mAP@50: {result["map_50"].item():.6f}')
    print('Per-class AP:')
    for class_id, class_name in names.items():
        print(f'  {class_name:18} {result["map_per_class"][class_id - 1].item():.6f}')

best_epoch = max(results, key=lambda epoch: results[epoch]['map'].item())
print(f'\nBest test mAP for comparison: Epoch {best_epoch}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/vehicle-damage-triage
Loaded project modules directly from absolute paths.

Evaluating Epoch 4 on untouched test2017...
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 174MB/s] 


Overall mAP@[0.50:0.95]: 0.406623
mAP@50: 0.627591
Per-class AP:
  dent               0.189447
  scratch            0.194205
  crack              0.145286
  glass shatter      0.738789
  lamp broken        0.472135
  tire flat          0.699874

Evaluating Epoch 5 on untouched test2017...
Overall mAP@[0.50:0.95]: 0.373025
mAP@50: 0.589070
Per-class AP:
  dent               0.127097
  scratch            0.213735
  crack              0.102535
  glass shatter      0.731312
  lamp broken        0.431956
  tire flat          0.631516

Best test mAP for comparison: Epoch 4
